# ETL Gold: Leitura, Resumo e Otimização dos Dados

In [0]:
# ========================================
# CAMADA GOLD - MODELO DIMENSIONAL
# ========================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Ler dados do Silver
df_silver = spark.table("acidentes.silver.acidentes_2025")

# Criar schema Gold se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS acidentes.gold")

print("📊 RESUMO DA CAMADA SILVER")
print("=" * 80)
print(f"Total de registros: {df_silver.count():,}")
print(f"\nColunas disponíveis: {len(df_silver.columns)}")
print("\n🔑 Principais dimensões identificadas:")
print("  • Tempo: data, hora, dia_semana, periodo_dia")
print("  • Localização: uf, br, municipio, km")
print("  • Condições da Via: tipo_pista, tracado_via, tipo_uso_solo")
print("  • Ambiente: condicao_metereologica")
print("  • Tipo de Acidente: tipo_acidente, causa_acidente, classificacao_acidente")
print("\n📈 Métricas: pessoas, mortos, feridos, ilesos, veiculos")
print("=" * 80)

In [0]:
display(df_silver.limit(20))

In [0]:
# ========================================
# CRIAR DATAFRAME OTIMIZADO PARA GOLD
# ========================================

# Selecionar apenas colunas necessárias
df_gold_base = df_silver.select(
    # Identificador
    col("id"),
    
    # Dimensão Tempo
    col("data"),
    col("ano"),
    col("mes"),
    col("dia_do_mes"),
    col("dia_semana"),
    col("hora"),
    col("periodo_dia"),
    
    # Dimensão Localização
    col("uf"),
    col("br"),
    col("municipio"),
    col("km"),
    
    # Dimensão Condições Via
    col("tipo_pista"),
    col("tracado_via"),
    col("tipo_uso_solo"),
    
    # Dimensão Ambiente
    col("condicao_metereologica"),
    
    # Dimensão Tipo Acidente
    col("tipo_acidente"),
    col("causa_acidente"),
    col("classificacao_acidente"),
    
    # Métricas
    col("pessoas"),
    col("mortos"),
    col("feridos"),
    col("ilesos"),
    col("ignorados"),
    col("veiculos")
)

print("✅ DataFrame otimizado criado!")
print(f"   Colunas: {len(df_silver.columns)} → {len(df_gold_base.columns)}")
print(f"   Registros: {df_gold_base.count():,}")

# Visualizar amostra
print("\n📋 AMOSTRA DOS DADOS:")
df_gold_base.display()

#Criação das tabelas de dimensões e fato

In [0]:
# ========================================
# DIMENSÃO TEMPO
# ========================================

dim_tempo = df_gold_base.select(
    concat(
        lpad(col("ano").cast("string"), 4, "0"),
        lpad(col("mes").cast("string"), 2, "0"),
        lpad(col("dia_do_mes").cast("string"), 2, "0"),
        lpad(col("hora").cast("string"), 2, "0")
    ).alias("tempo_sk"),
    col("data"),
    col("ano"),
    col("mes"),
    col("dia_do_mes"),
    col("dia_semana"),
    col("hora"),
    col("periodo_dia"),
    # Flags úteis para análise
    when(col("dia_semana").isin("sábado", "domingo"), True).otherwise(False).alias("is_final_semana"),
    when(col("hora").between(6, 11), "Manhã")
        .when(col("hora").between(12, 17), "Tarde")
        .when(col("hora").between(18, 23), "Noite")
        .otherwise("Madrugada").alias("turno"),
    # Trimestre
    when(col("mes").between(1, 3), 1)
        .when(col("mes").between(4, 6), 2)
        .when(col("mes").between(7, 9), 3)
        .otherwise(4).alias("trimestre")
).distinct()

dim_tempo.write.mode("overwrite").saveAsTable("acidentes.gold.dim_tempo")
print(f"✅ dim_tempo criada: {dim_tempo.count():,} registros")

In [0]:
# ========================================
# DIMENSÃO LOCALIZAÇÃO
# ========================================

dim_localizacao = df_gold_base.select(
    md5(concat_ws("_", col("uf"), col("br"), col("municipio"))).alias("localizacao_sk"),
    col("uf"),
    col("br").alias("rodovia_br"),
    col("municipio"),
    # Região do Brasil
    when(col("uf").isin("SP", "RJ", "MG", "ES"), "Sudeste")
        .when(col("uf").isin("PR", "SC", "RS"), "Sul")
        .when(col("uf").isin("BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"), "Nordeste")
        .when(col("uf").isin("GO", "MT", "MS", "DF"), "Centro-Oeste")
        .when(col("uf").isin("AM", "RR", "AP", "PA", "TO", "RO", "AC"), "Norte")
        .otherwise("Não identificado").alias("regiao")
).distinct()

dim_localizacao.write.mode("overwrite").saveAsTable("acidentes.gold.dim_localizacao")
print(f"✅ dim_localizacao criada: {dim_localizacao.count():,} registros")

In [0]:
# ========================================
# DIMENSÃO CONDIÇÕES DA VIA
# ========================================

dim_condicoes_via = df_gold_base.select(
    md5(concat_ws("_", col("tipo_pista"), col("tracado_via"), col("tipo_uso_solo"))).alias("condicoes_via_sk"),
    col("tipo_pista"),
    col("tracado_via"),
    col("tipo_uso_solo"),
    # Classificação de risco da pista
    when(col("tipo_pista").like("%Dupla%"), "Baixo")
        .when(col("tipo_pista").like("%Simples%"), "Médio")
        .when(col("tipo_pista").like("%Múltipla%"), "Baixo")
        .otherwise("Desconhecido").alias("nivel_risco_pista")
).distinct()

dim_condicoes_via.write.mode("overwrite").saveAsTable("acidentes.gold.dim_condicoes_via")
print(f"✅ dim_condicoes_via criada: {dim_condicoes_via.count():,} registros")

In [0]:
# ========================================
# DIMENSÃO AMBIENTE
# ========================================

dim_ambiente = df_gold_base.select(
    md5(col("condicao_metereologica")).alias("ambiente_sk"),
    col("condicao_metereologica"),
    # Classificação simplificada
    when(col("condicao_metereologica").like("%Chuva%"), "Adverso")
        .when(col("condicao_metereologica").like("%Nevoeiro%"), "Adverso")
        .when(col("condicao_metereologica").like("%Nublado%"), "Adverso")
        .when(col("condicao_metereologica").like("%Céu Claro%"), "Normal")
        .when(col("condicao_metereologica").like("%Sol%"), "Normal")
        .otherwise("Outro").alias("tipo_condicao")
).distinct()

dim_ambiente.write.mode("overwrite").saveAsTable("acidentes.gold.dim_ambiente")
print(f"✅ dim_ambiente criada: {dim_ambiente.count():,} registros")

In [0]:
# ========================================
# DIMENSÃO TIPO ACIDENTE
# ========================================

dim_tipo_acidente = df_gold_base.select(
    md5(concat_ws("_", col("tipo_acidente"), col("causa_acidente"), col("classificacao_acidente"))).alias("tipo_acidente_sk"),
    col("tipo_acidente"),
    col("causa_acidente"),
    col("classificacao_acidente"),
    # Indicador de gravidade
    when(col("classificacao_acidente").like("%Mortes%"), "Fatal")
        .when(col("classificacao_acidente").like("%Com Vítimas Fatais%"), "Fatal")
        .when(col("classificacao_acidente").like("%Feridos Graves%"), "Grave")
        .when(col("classificacao_acidente").like("%Feridos%"), "Leve")
        .otherwise("Sem Vítimas").alias("nivel_gravidade")
).distinct()

dim_tipo_acidente.write.mode("overwrite").saveAsTable("acidentes.gold.dim_tipo_acidente")
print(f"✅ dim_tipo_acidente criada: {dim_tipo_acidente.count():,} registros")

In [0]:
# ========================================
# TABELA FATO - ACIDENTES
# ========================================

fato_acidentes = df_gold_base.select(
    col("id").alias("acidente_id"),
    
    # Chaves estrangeiras (FKs) para as dimensões
    concat(
        lpad(col("ano").cast("string"), 4, "0"),
        lpad(col("mes").cast("string"), 2, "0"),
        lpad(col("dia_do_mes").cast("string"), 2, "0"),
        lpad(col("hora").cast("string"), 2, "0")
    ).alias("tempo_sk"),
    
    md5(concat_ws("_", col("uf"), col("br"), col("municipio"))).alias("localizacao_sk"),
    md5(concat_ws("_", col("tipo_pista"), col("tracado_via"), col("tipo_uso_solo"))).alias("condicoes_via_sk"),
    md5(col("condicao_metereologica")).alias("ambiente_sk"),
    md5(concat_ws("_", col("tipo_acidente"), col("causa_acidente"), col("classificacao_acidente"))).alias("tipo_acidente_sk"),
    
    # Métricas numéricas
    col("pessoas").alias("qtd_pessoas"),
    col("mortos").alias("qtd_mortos"),
    col("feridos").alias("qtd_feridos"),
    col("ilesos").alias("qtd_ilesos"),
    col("ignorados").alias("qtd_ignorados"),
    col("veiculos").alias("qtd_veiculos"),
    col("km"),
    
    # Métricas calculadas
    (col("mortos") + col("feridos")).alias("qtd_vitimas"),
    when(col("mortos") > 0, 1).otherwise(0).alias("flag_fatal"),
    when(col("feridos") > 0, 1).otherwise(0).alias("flag_com_feridos"),
    
    # Indicador de gravidade numérico (para análises)
    (col("mortos") * 10 + col("feridos")).alias("indice_gravidade")
)

fato_acidentes.write.mode("overwrite").saveAsTable("acidentes.gold.fato_acidentes")
print(f"✅ fato_acidentes criada: {fato_acidentes.count():,} registros")

In [0]:
# ========================================
# RESUMO DAS TABELAS CRIADAS
# ========================================

print("\n" + "=" * 80)
print("📊 CAMADA GOLD - MODELO DIMENSIONAL CRIADO")
print("=" * 80)

tabelas_gold = [
    ("dim_tempo", "acidentes.gold.dim_tempo"),
    ("dim_localizacao", "acidentes.gold.dim_localizacao"),
    ("dim_condicoes_via", "acidentes.gold.dim_condicoes_via"),
    ("dim_ambiente", "acidentes.gold.dim_ambiente"),
    ("dim_tipo_acidente", "acidentes.gold.dim_tipo_acidente"),
    ("fato_acidentes", "acidentes.gold.fato_acidentes")
]

for nome, tabela in tabelas_gold:
    count = spark.table(tabela).count()
    print(f"  ✅ {nome:25s}: {count:>10,} registros")

print("=" * 80)
print("✅ Modelo Star Schema completo!")

# ANÁLISES E VISUALIZAÇÕES - FOCO EM FATALIDADES

In [0]:
# ========================================
# 📊 CARREGAR DADOS - RODOVIAS
# ========================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Buscar dados das rodovias mais perigosas
df_rodovias = spark.sql("""
    SELECT 
        l.rodovia_br,
        COUNT(*) as total_acidentes,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
        SUM(f.flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade,
        ROUND(100.0 * SUM(f.flag_fatal) / COUNT(*), 2) as perc_acidentes_fatais,
        ROUND(1.0 * SUM(f.qtd_mortos) / COUNT(*), 3) as taxa_mortalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_localizacao l ON f.localizacao_sk = l.localizacao_sk
    WHERE f.flag_fatal = 1
    GROUP BY l.rodovia_br
    HAVING total_acidentes >= 10
    ORDER BY total_mortos DESC
    LIMIT 20
""").toPandas()

for col in ['taxa_letalidade', 'taxa_mortalidade']:
    df_rodovias[col] = df_rodovias[col].astype(float)

df_rodovias['rodovia'] = 'BR-' + df_rodovias['rodovia_br'].astype(str)

print("✅ Dados carregados: Top 20 Rodovias (números absolutos)")
print(f"Total de rodovias analisadas: {len(df_rodovias)}")

In [0]:
# ========================================
# ☠️ GRÁFICO: TOP 20 RODOVIAS COM MAIS MORTES
# ========================================

fig, ax = plt.subplots(figsize=(12, 10))

colors = plt.cm.Reds(
    np.linspace(0.4, 0.95, len(df_rodovias))
)

bars = ax.barh(
    df_rodovias['rodovia'],
    df_rodovias['total_mortos'],
    color=colors
)

ax.invert_yaxis()

max_mortos = df_rodovias['total_mortos'].max()
ax.set_xlim(0, max_mortos * 1.15)

for bar in bars:

    valor = bar.get_width()

    ax.text(
        valor + (max_mortos * 0.01),
        bar.get_y() + bar.get_height()/2,
        f'{int(valor)} mortos',
        va='center',
        ha='left',
        fontsize=9,
        fontweight='bold'
    )

ax.set_xlabel(
    'Total de Mortos',
    fontsize=12,
    fontweight='bold'
)

ax.set_title(
    '☠️ Top 20 Rodovias com Mais Mortes (Números Absolutos)\n'
    '(Apenas Acidentes Fatais)',
    fontsize=14,
    fontweight='bold'
)

ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n☠️ RANKING: RODOVIAS COM MAIS MORTOS (NÚMEROS ABSOLUTOS):")
print(
    df_rodovias[
        [
            'rodovia',
            'total_mortos',
            'acidentes_fatais',
            'total_pessoas_envolvidas',
            'taxa_letalidade'
        ]
    ].head(10).to_string(index=False)
)

In [0]:
# ========================================
# 📅 ANÁLISE 2: DIA DA SEMANA x HORA (MAPA DE CALOR DE MORTES)
# ========================================

# Buscar dados de mortes por dia e hora
df_dia_hora = spark.sql("""
    SELECT 
        t.dia_semana,
        t.hora,
        SUM(f.qtd_mortos) as total_mortos,
        COUNT(*) as total_acidentes,
        SUM(f.flag_fatal) as acidentes_fatais
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tempo t ON f.tempo_sk = t.tempo_sk
    WHERE f.flag_fatal = 1  -- APENAS ACIDENTES FATAIS
    GROUP BY t.dia_semana, t.hora
""").toPandas()

# Ordenar dias da semana corretamente
order_dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado', 'domingo']
df_dia_hora['dia_semana'] = pd.Categorical(df_dia_hora['dia_semana'], categories=order_dias, ordered=True)
df_dia_hora = df_dia_hora.sort_values(['dia_semana', 'hora'])

# Criar matriz pivot
heatmap_mortos = df_dia_hora.pivot(index='dia_semana', columns='hora', values='total_mortos').fillna(0)

# Criar visualização
fig, ax = plt.subplots(figsize=(22, 8))
sns.heatmap(heatmap_mortos, annot=True, fmt='.0f', cmap='YlOrRd', 
            linewidths=0.5, cbar_kws={'label': 'Total de Mortos'},
            ax=ax, vmin=0)

ax.set_title('🔥 MAPA DE CALOR: Mortes por Dia da Semana e Hora\n(Apenas Acidentes Fatais - Quanto mais escuro, mais mortes)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Hora do Dia', fontsize=13, fontweight='bold')
ax.set_ylabel('Dia da Semana', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Identificar horários mais perigosos
print("\n⚠️ TOP 15 COMBINAÇÕES MAIS LETAIS (Dia + Hora):")
top_combinacoes = df_dia_hora.nlargest(15, 'total_mortos')[['dia_semana', 'hora', 'total_mortos', 'acidentes_fatais']]
print(top_combinacoes.to_string(index=False))

In [0]:
# ========================================
# ☁️ CARREGAR DADOS - CONDIÇÕES METEOROLÓGICAS
# ========================================

df_meteo = spark.sql("""
    SELECT 
        amb.condicao_metereologica,
        COUNT(*) as total_acidentes_fatais,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade,
        ROUND(AVG(f.qtd_mortos), 2) as media_mortos_acidente,
        ROUND(AVG(f.qtd_feridos), 2) as media_feridos_acidente,
        ROUND(100.0 * SUM(f.qtd_mortos) / SUM(SUM(f.qtd_mortos)) OVER (), 2) as perc_total_mortes
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_ambiente amb ON f.ambiente_sk = amb.ambiente_sk
    WHERE f.flag_fatal = 1
    GROUP BY amb.condicao_metereologica
    ORDER BY taxa_letalidade DESC
""").toPandas()

for col in ['taxa_letalidade', 'media_mortos_acidente', 'media_feridos_acidente', 'perc_total_mortes']:
    df_meteo[col] = df_meteo[col].astype(float)

print("✅ Dados carregados: Condições Meteorológicas")
print(f"Total de condições analisadas: {len(df_meteo)}")

In [0]:
# ========================================
# ☁️ GRÁFICO 3A: TAXA DE LETALIDADE POR CLIMA
# ========================================
fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(df_meteo)))
ax.barh(df_meteo['condicao_metereologica'], df_meteo['taxa_letalidade'], color=colors)
ax.set_xlabel('Taxa de Letalidade (%)', fontsize=12, fontweight='bold')
ax.set_title('☁️ Taxa de Letalidade por Condição Meteorológica\n(% de pessoas que morreram vs total de envolvidos)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

for i, (taxa, pessoas) in enumerate(zip(df_meteo['taxa_letalidade'], df_meteo['total_pessoas_envolvidas'])):
    ax.text(float(taxa) + 0.5, i, f'{taxa:.1f}% ({int(pessoas)} pessoas)', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n☁️ RANKING POR TAXA DE LETALIDADE:")
print(df_meteo[['condicao_metereologica', 'taxa_letalidade', 'total_pessoas_envolvidas', 'total_mortos', 'total_acidentes_fatais']].to_string(index=False))

In [0]:
# ========================================
# ☁️ GRÁFICO 3B: FREQUÊNCIA vs LETALIDADE
# ========================================
fig, ax1 = plt.subplots(figsize=(12, 7))

# Ordenar por frequência (total de acidentes)
df_meteo_plot = df_meteo.sort_values('total_acidentes_fatais', ascending=True)

# Eixo 1: Barras de frequência (número de acidentes)
color_barras = 'steelblue'
ax1.barh(df_meteo_plot['condicao_metereologica'], df_meteo_plot['total_acidentes_fatais'], 
         color=color_barras, alpha=0.6, label='Frequência de Acidentes')
ax1.set_xlabel('Número de Acidentes Fatais', fontsize=11, fontweight='bold', color=color_barras)
ax1.tick_params(axis='x', labelcolor=color_barras)
ax1.grid(axis='x', alpha=0.3)

# Eixo 2: Linha de letalidade (%)
ax2 = ax1.twiny()
color_linha = 'darkred'
ax2.plot(df_meteo_plot['taxa_letalidade'], range(len(df_meteo_plot)), 
         color=color_linha, marker='o', linewidth=2.5, markersize=8, label='Taxa de Letalidade (%)')
ax2.set_xlabel('Taxa de Letalidade (%)', fontsize=11, fontweight='bold', color=color_linha)
ax2.tick_params(axis='x', labelcolor=color_linha)

# Adicionar valores nas barras e pontos
for i, (acidentes, taxa) in enumerate(zip(df_meteo_plot['total_acidentes_fatais'], df_meteo_plot['taxa_letalidade'])):
    # Valor da barra (frequência)
    ax1.text(acidentes + 20, i, f'{int(acidentes)}', va='center', fontsize=9, color=color_barras, fontweight='bold')
    # Valor da linha (letalidade)
    ax2.text(taxa + 0.3, i, f'{taxa:.1f}%', va='center', fontsize=9, color=color_linha, fontweight='bold')

ax1.set_title('☁️ Frequência vs Letalidade por Condição Meteorológica\n(Condições raras podem ser mais letais!)', 
              fontsize=13, fontweight='bold', pad=20)

# Legendas
ax1.legend(loc='upper left', fontsize=10)
ax2.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print("\n☁️ ANÁLISE COMPARATIVA - FREQUÊNCIA vs LETALIDADE:")
print(df_meteo_plot[['condicao_metereologica', 'total_acidentes_fatais', 'taxa_letalidade', 'total_mortos']].to_string(index=False))

In [0]:
# ========================================
# 🚗 CARREGAR DADOS - CAUSAS DE ACIDENTES
# ========================================

# Query 1: Top 15 CAUSAS por VOLUME (números absolutos)
df_causas_volume = spark.sql("""
    SELECT
        ta.causa_acidente,
        COUNT(*) as total_acidentes_fatais,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,

        ROUND(
            100.0 * SUM(f.qtd_mortos) /
            NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0),
            2
        ) as taxa_letalidade

    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tipo_acidente ta
        ON f.tipo_acidente_sk = ta.tipo_acidente_sk

    WHERE f.flag_fatal = 1

    GROUP BY ta.causa_acidente

    ORDER BY total_mortos DESC

    LIMIT 15
""").toPandas()

for col in ['taxa_letalidade']:
    df_causas_volume[col] = df_causas_volume[col].astype(float)

# Query 2: Top 15 CAUSAS por LETALIDADE (% mortos/envolvidos)
df_causas_letalidade = spark.sql("""
    SELECT 
        ta.causa_acidente,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade,
        COUNT(*) as total_acidentes_fatais
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
    WHERE f.flag_fatal = 1
    GROUP BY ta.causa_acidente
    ORDER BY taxa_letalidade DESC
    LIMIT 15
""").toPandas()

for col in ['taxa_letalidade']:
    df_causas_letalidade[col] = df_causas_letalidade[col].astype(float)

print("✅ Dados carregados: Causas de Acidentes")
print(f"Total de causas por volume: {len(df_causas_volume)}")
print(f"Total de causas por letalidade: {len(df_causas_letalidade)}")

In [0]:
# ========================================
# 🚗 GRÁFICO 4A: TOP 15 CAUSAS POR VOLUME
# ========================================
fig, ax = plt.subplots(figsize=(14, 10))

colors = plt.cm.Reds(
    np.linspace(0.4, 0.95, len(df_causas_volume))
)

bars = ax.barh(
    df_causas_volume['causa_acidente'],
    df_causas_volume['total_mortos'],
    color=colors
)

ax.invert_yaxis()

max_mortos = df_causas_volume['total_mortos'].max()

for bar in bars:

    valor = bar.get_width()

    ax.text(
        valor + max_mortos * 0.01,
        bar.get_y() + bar.get_height()/2,
        f'{int(valor)} mortos',
        va='center',
        fontsize=10,
        fontweight='bold'
    )

ax.set_xlim(0, max_mortos * 1.25)

ax.set_xlabel(
    'Total de Mortos',
    fontsize=12,
    fontweight='bold'
)

ax.set_title(
    '☠️ Top 15 Causas com Mais Mortes (Números Absolutos)\n'
    '(Apenas Acidentes Fatais)',
    fontsize=14,
    fontweight='bold'
)

ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n☠️ TOP 15 CAUSAS COM MAIS MORTES (NÚMEROS ABSOLUTOS):")
print(df_causas_volume[['causa_acidente', 'total_mortos', 'total_acidentes_fatais', 'taxa_letalidade']].to_string(index=False))

In [0]:
# ========================================
# 🚗 GRÁFICO 4B: TOP 15 CAUSAS POR LETALIDADE
# ========================================

fig, ax = plt.subplots(figsize=(14, 10))

colors = plt.cm.Oranges(np.linspace(0.4, 0.95, len(df_causas_letalidade)))
ax.barh(df_causas_letalidade['causa_acidente'], df_causas_letalidade['taxa_letalidade'], color=colors)
ax.set_xlabel('Taxa de Letalidade (%)', fontsize=12, fontweight='bold')
ax.set_title('⚠️ Top 15 Causas Mais Letais\n(Ordenado por % de letalidade - sem filtro de volume)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Adicionar valores
for i, (taxa, pessoas) in enumerate(zip(df_causas_letalidade['taxa_letalidade'], df_causas_letalidade['total_pessoas_envolvidas'])):
    ax.text(float(taxa) + 0.5, i, f'{taxa:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ TOP 15 CAUSAS MAIS LETAIS:")
print(df_causas_letalidade[['causa_acidente', 'taxa_letalidade', 'total_pessoas_envolvidas', 'total_mortos', 'total_acidentes_fatais']].to_string(index=False))

In [0]:
# ========================================
# 🚗 GRÁFICO 4B-2: TOP 15 TIPOS MAIS LETAIS
# ========================================

# Carregar dados de tipos por letalidade
df_tipos_letalidade = spark.sql("""
    SELECT 
        ta.tipo_acidente,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade,
        COUNT(*) as total_acidentes_fatais
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
    WHERE f.flag_fatal = 1
    GROUP BY ta.tipo_acidente
    ORDER BY taxa_letalidade DESC
    LIMIT 15
""").toPandas()

for col in ['taxa_letalidade']:
    df_tipos_letalidade[col] = df_tipos_letalidade[col].astype(float)

# Criar gráfico
fig, ax = plt.subplots(figsize=(14, 10))

colors = plt.cm.Purples(np.linspace(0.4, 0.95, len(df_tipos_letalidade)))
ax.barh(df_tipos_letalidade['tipo_acidente'], df_tipos_letalidade['taxa_letalidade'], color=colors)
ax.set_xlabel('Taxa de Letalidade (%)', fontsize=12, fontweight='bold')
ax.set_title('⚠️ Top 15 Tipos de Acidente Mais Letais\n(Ordenado por % de letalidade - sem filtro de volume)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Adicionar valores
for i, taxa in enumerate(df_tipos_letalidade['taxa_letalidade']):
    ax.text(float(taxa) + 0.5, i, f'{taxa:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ TOP 15 TIPOS DE ACIDENTE MAIS LETAIS:")
print(df_tipos_letalidade[['tipo_acidente', 'taxa_letalidade', 'total_pessoas_envolvidas', 'total_mortos', 'total_acidentes_fatais']].to_string(index=False))

In [0]:
# ========================================
# 🚗 GRÁFICO 4C: FREQUÊNCIA vs LETALIDADE
# ========================================

# Pegar top 15 causas por frequência para comparar
df_causas_comparativo = spark.sql("""
    SELECT 
        ta.causa_acidente,
        COUNT(*) as total_acidentes_fatais,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
    WHERE f.flag_fatal = 1
    GROUP BY ta.causa_acidente
    ORDER BY total_acidentes_fatais DESC
    LIMIT 15
""").toPandas()

for col in ['taxa_letalidade']:
    df_causas_comparativo[col] = df_causas_comparativo[col].astype(float)

# Ordenar por frequência
df_causas_comparativo = df_causas_comparativo.sort_values('total_acidentes_fatais', ascending=True)

fig, ax1 = plt.subplots(figsize=(14, 10))

# Eixo 1: Barras de frequência (número de acidentes)
color_barras = 'steelblue'
ax1.barh(df_causas_comparativo['causa_acidente'], df_causas_comparativo['total_acidentes_fatais'], 
         color=color_barras, alpha=0.6, label='Frequência de Acidentes')
ax1.set_xlabel('Número de Acidentes Fatais', fontsize=11, fontweight='bold', color=color_barras)
ax1.tick_params(axis='x', labelcolor=color_barras)
ax1.grid(axis='x', alpha=0.3)

# Eixo 2: Linha de letalidade (%)
ax2 = ax1.twiny()
color_linha = 'darkred'
ax2.plot(df_causas_comparativo['taxa_letalidade'], range(len(df_causas_comparativo)), 
         color=color_linha, marker='o', linewidth=2.5, markersize=8, label='Taxa de Letalidade (%)')
ax2.set_xlabel('Taxa de Letalidade (%)', fontsize=11, fontweight='bold', color=color_linha)
ax2.tick_params(axis='x', labelcolor=color_linha)

# Adicionar valores nas barras e pontos
for i, (acidentes, taxa) in enumerate(zip(df_causas_comparativo['total_acidentes_fatais'], df_causas_comparativo['taxa_letalidade'])):
    # Valor da barra (frequência)
    ax1.text(acidentes + 5, i, f'{int(acidentes)}', va='center', fontsize=9, color=color_barras, fontweight='bold')
    # Valor da linha (letalidade) - com fundo branco para melhor legibilidade
    ax2.text(taxa + 1.0, i, f'{taxa:.1f}%', va='center', fontsize=9, color=color_linha, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='none', alpha=0.7))

ax1.set_title('🚗 Frequência vs Letalidade por Causa de Acidente\n(Causas frequentes nem sempre são as mais letais!)', 
              fontsize=14, fontweight='bold', pad=20)

# Legendas
ax1.legend(loc='upper left', fontsize=10)
ax2.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🚗 ANÁLISE COMPARATIVA - FREQUÊNCIA vs LETALIDADE:")
print(df_causas_comparativo[['causa_acidente', 'total_acidentes_fatais', 'taxa_letalidade', 'total_mortos']].to_string(index=False))

In [0]:
# ========================================
# 🚗 GRÁFICO 4D: ASSOCIAÇÃO TIPO x CAUSA (LINHAS)
# ========================================

# Buscar top 10 causas e seus tipos associados
df_tipo_causa_linhas = spark.sql("""
    SELECT 
        ta.causa_acidente,
        ta.tipo_acidente,
        COUNT(*) as total_acidentes_fatais,
        SUM(f.qtd_mortos) as total_mortos
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
    WHERE f.flag_fatal = 1
    GROUP BY ta.causa_acidente, ta.tipo_acidente
    ORDER BY total_mortos DESC
""").toPandas()

# Pegar top 10 causas por volume total
top_causas_list = df_tipo_causa_linhas.groupby('causa_acidente')['total_mortos'].sum().nlargest(10).index.tolist()

# Filtrar apenas essas causas
df_plot = df_tipo_causa_linhas[df_tipo_causa_linhas['causa_acidente'].isin(top_causas_list)]

# Pegar top 5 tipos mais frequentes
top_tipos = df_plot.groupby('tipo_acidente')['total_mortos'].sum().nlargest(5).index.tolist()

# Criar gráfico
fig, ax = plt.subplots(figsize=(16, 10))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
markers = ['o', 's', '^', 'D', 'v']

# Para cada tipo, criar uma linha
for i, tipo in enumerate(top_tipos):
    df_tipo = df_plot[df_plot['tipo_acidente'] == tipo]
    
    # Ordenar por ordem das top causas
    df_tipo = df_tipo.set_index('causa_acidente').reindex(top_causas_list).reset_index()
    df_tipo['total_mortos'] = df_tipo['total_mortos'].fillna(0)
    
    ax.plot(range(len(top_causas_list)), df_tipo['total_mortos'], 
            marker=markers[i], linewidth=2.5, markersize=8, 
            color=colors[i], label=tipo[:30], alpha=0.8)

ax.set_xticks(range(len(top_causas_list)))
ax.set_xticklabels(top_causas_list, rotation=45, ha='right', fontsize=10)
ax.set_xlabel('Causa do Acidente', fontsize=12, fontweight='bold')
ax.set_ylabel('Total de Mortos', fontsize=12, fontweight='bold')
ax.set_title('🚗 Associação: Tipos de Acidente x Causas\n(Top 10 Causas x Top 5 Tipos mais letais)', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10, title='Tipo de Acidente', title_fontsize=11)

plt.tight_layout()
plt.show()

print("\n🚗 TABELA: TIPOS x CAUSAS (Top 10 causas x Top 5 tipos):")
df_pivot = df_plot[df_plot['tipo_acidente'].isin(top_tipos)].pivot_table(
    index='causa_acidente', 
    columns='tipo_acidente', 
    values='total_mortos', 
    fill_value=0
).reindex(top_causas_list)

print(df_pivot.to_string())

In [0]:
# ========================================
# 📆 ANÁLISE 5: MÊS x FATALIDADES (SAZONALIDADE)
# ========================================

# Buscar dados por mês
df_mes = spark.sql("""
    SELECT 
        t.mes,
        CASE t.mes
            WHEN 1 THEN 'Janeiro'
            WHEN 2 THEN 'Fevereiro'
            WHEN 3 THEN 'Março'
            WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Maio'
            WHEN 6 THEN 'Junho'
            WHEN 7 THEN 'Julho'
            WHEN 8 THEN 'Agosto'
            WHEN 9 THEN 'Setembro'
            WHEN 10 THEN 'Outubro'
            WHEN 11 THEN 'Novembro'
            WHEN 12 THEN 'Dezembro'
        END as mes_nome,
        COUNT(*) as total_acidentes,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.qtd_feridos) as total_feridos,
        SUM(f.flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(f.flag_fatal) / COUNT(*), 2) as perc_acidentes_fatais,
        ROUND(1.0 * SUM(f.qtd_mortos) / COUNT(*), 3) as taxa_mortalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tempo t ON f.tempo_sk = t.tempo_sk
    GROUP BY t.mes
    ORDER BY t.mes
""").toPandas()

# Corrigir tipos Decimal vs float
for col in ["perc_acidentes_fatais", "taxa_mortalidade"]:
    df_mes[col] = df_mes[col].astype(float)

print("✅ Dados carregados: Análise Mensal")
print(f"Total de meses analisados: {len(df_mes)}")

In [0]:
# Gráfico 5A: Linha - Total de Mortos por Mês
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_mes['mes_nome'], df_mes['total_mortos'], marker='o', linewidth=2.5, 
        markersize=8, color='darkred', label='Total de Mortos')
ax.fill_between(range(len(df_mes)), df_mes['total_mortos'], alpha=0.3, color='red')
ax.set_xlabel('Mês', fontsize=11, fontweight='bold')
ax.set_ylabel('Total de Mortos', fontsize=11, fontweight='bold')
ax.set_title('☠️ Mortes ao Longo do Ano (2025)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

for i, (mes, mortos) in enumerate(zip(df_mes['mes_nome'], df_mes['total_mortos'])):
    ax.text(i, mortos + 10, str(int(mortos)), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📆 MORTES POR MÊS:")
print(df_mes[['mes_nome', 'total_mortos', 'total_acidentes']].to_string(index=False))

In [0]:
# Gráfico 5B: Barras - Acidentes Fatais por Mês
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(df_mes)))
ax.bar(df_mes['mes_nome'], df_mes['acidentes_fatais'], color=colors)
ax.set_xlabel('Mês', fontsize=11, fontweight='bold')
ax.set_ylabel('Acidentes Fatais', fontsize=11, fontweight='bold')
ax.set_title('🚨 Acidentes Fatais por Mês', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

for i, v in enumerate(df_mes['acidentes_fatais']):
    ax.text(i, v + 5, str(int(v)), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🚨 ACIDENTES FATAIS POR MÊS:")
print(df_mes[['mes_nome', 'acidentes_fatais', 'perc_acidentes_fatais']].to_string(index=False))

In [0]:
# Gráfico 5C: Taxa de Mortalidade por Mês
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(df_mes)))
ax.bar(df_mes['mes_nome'], df_mes['taxa_mortalidade'], color=colors)
ax.set_xlabel('Mês', fontsize=11, fontweight='bold')
ax.set_ylabel('Taxa de Mortalidade (mortos/acidente)', fontsize=11, fontweight='bold')
ax.set_title('📊 Taxa de Mortalidade por Mês', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

for i, v in enumerate(df_mes['taxa_mortalidade']):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 TAXA DE MORTALIDADE POR MÊS:")
print(df_mes[['mes_nome', 'taxa_mortalidade', 'total_mortos', 'total_acidentes']].to_string(index=False))

In [0]:
# Gráfico 5D: Tabela Resumo Mensal
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

table_data = []
for _, row in df_mes.iterrows():
    table_data.append([
        row['mes_nome'][:3],
        f"{int(row['total_acidentes'])}",
        f"{int(row['total_mortos'])}",
        f"{int(row['acidentes_fatais'])}",
        f"{row['perc_acidentes_fatais']:.1f}%",
        f"{row['taxa_mortalidade']:.3f}"
    ])

table = ax.table(cellText=table_data,
                  colLabels=['Mês', 'Acidentes', 'Mortos', 'Fatais', '% Fatal', 'Taxa Mort.'],
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.12, 0.18, 0.18, 0.18, 0.16, 0.18])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

for i in range(6):
    table[(0, i)].set_facecolor('#2E75B6')
    table[(0, i)].set_text_props(weight='bold', color='white')

ax.set_title('📋 Resumo Mensal Completo', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n📆 ANÁLISE MENSAL COMPLETA:")
print(df_mes[['mes_nome', 'total_acidentes', 'total_mortos', 'acidentes_fatais', 'perc_acidentes_fatais', 'taxa_mortalidade']].to_string(index=False))

In [0]:
# ========================================
# 🎯 RESUMO EXECUTIVO - KPIs DE FATALIDADES
# ========================================

print("=" * 100)
print("🎯 RESUMO EXECUTIVO - ANÁLISE DE FATALIDADES EM RODOVIAS FEDERAIS (2025)")
print("=" * 100)

# KPIs Gerais
kpis = spark.sql("""
    SELECT 
        COUNT(*) as total_acidentes,
        SUM(qtd_mortos) as total_mortos,
        SUM(qtd_feridos) as total_feridos,
        SUM(flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(flag_fatal) / COUNT(*), 2) as perc_fatais,
        ROUND(100.0 * SUM(qtd_mortos) / SUM(qtd_mortos + qtd_feridos), 2) as perc_mortos,
        ROUND(1.0 * SUM(qtd_mortos) / SUM(flag_fatal), 2) as media_mortos_por_acidente_fatal
    FROM acidentes.gold.fato_acidentes
""").collect()[0]

print(f"\n📊 KPIs GERAIS:")
print(f"   • Total de Acidentes: {kpis['total_acidentes']:,}")
print(f"   • Total de Mortos: {kpis['total_mortos']:,} ⚠️ ({kpis['perc_mortos']}% dos envolvidos)")
print(f"   • Total de Feridos: {kpis['total_feridos']:,}")
print(f"   • Acidentes Fatais: {kpis['acidentes_fatais']:,} ({kpis['perc_fatais']}%)")
print(f"   • Média de Mortos por Acidente Fatal: {kpis['media_mortos_por_acidente_fatal']}")

# Top 3 Rodovias Mais Perigosas
print(f"\n\n🚨 TOP 3 RODOVIAS MAIS PERIGOSAS:")
top_rodovias = spark.sql("""
    SELECT 
        CONCAT('BR-', l.rodovia_br) as rodovia,
        l.regiao,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_localizacao l ON f.localizacao_sk = l.localizacao_sk
    WHERE f.flag_fatal = 1
    GROUP BY l.rodovia_br, l.regiao
    ORDER BY total_mortos DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_rodovias, 1):
    print(f"   {i}. {row['rodovia']} ({row['regiao']}): {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes fatais")

# Horários Mais Perigosos
print(f"\n\n🕒 TOP 3 HORÁRIOS MAIS PERIGOSOS:")
top_horarios = spark.sql("""
    SELECT 
        t.hora,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tempo t ON f.tempo_sk = t.tempo_sk
    WHERE f.flag_fatal = 1
    GROUP BY t.hora
    ORDER BY total_mortos DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_horarios, 1):
    print(f"   {i}. {row['hora']:02d}:00h - {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes fatais")

# Dias Mais Perigosos
print(f"\n\n📅 TOP 3 DIAS DA SEMANA MAIS PERIGOSOS:")
top_dias = spark.sql("""
    SELECT 
        t.dia_semana,
        SUM(f.qtd_mortos) as total_mortos,
        SUM(f.flag_fatal) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tempo t ON f.tempo_sk = t.tempo_sk
    WHERE f.flag_fatal = 1
    GROUP BY t.dia_semana
    ORDER BY total_mortos DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_dias, 1):
    print(f"   {i}. {row['dia_semana'].title()}: {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes fatais")

# TOP 3 COMBINAÇÕES TIPO x CAUSA MAIS LETAIS
print(f"\n\n🔥 TOP 3 COMBINAÇÕES MAIS LETAIS (TIPO x CAUSA):")
top_combinacoes = spark.sql("""
    WITH combinacoes AS (
        SELECT 
            ta.tipo_acidente,
            ta.causa_acidente,
            SUM(f.qtd_mortos) as total_mortos,
            SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
            COUNT(*) as acidentes_fatais,
            ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
        FROM acidentes.gold.fato_acidentes f
        INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
        WHERE f.flag_fatal = 1
        GROUP BY ta.tipo_acidente, ta.causa_acidente
        HAVING COUNT(*) >= 5 AND SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) >= 50
    )
    SELECT * FROM combinacoes
    ORDER BY taxa_letalidade DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_combinacoes, 1):
    tipo_short = row['tipo_acidente'][:35] + '...' if len(row['tipo_acidente']) > 35 else row['tipo_acidente']
    causa_short = row['causa_acidente'][:40] + '...' if len(row['causa_acidente']) > 40 else row['causa_acidente']
    print(f"   {i}. {tipo_short} + {causa_short}")
    print(f"      {row['total_mortos']} mortos / {row['total_pessoas_envolvidas']} pessoas = {row['taxa_letalidade']}% de letalidade")
    print(f"      ({row['acidentes_fatais']} acidentes fatais)")

# Principais Causas (ordenado por letalidade, não por total bruto)
print(f"\n\n🚗 TOP 3 CAUSAS MAIS LETAIS:")
top_causas = spark.sql("""
    WITH causas AS (
        SELECT 
            ta.causa_acidente,
            SUM(f.qtd_mortos) as total_mortos,
            SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) as total_pessoas_envolvidas,
            COUNT(*) as acidentes_fatais,
            ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
        FROM acidentes.gold.fato_acidentes f
        INNER JOIN acidentes.gold.dim_tipo_acidente ta ON f.tipo_acidente_sk = ta.tipo_acidente_sk
        WHERE f.flag_fatal = 1
        GROUP BY ta.causa_acidente
        HAVING COUNT(*) >= 10 AND SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos) >= 100
    )
    SELECT * FROM causas
    ORDER BY taxa_letalidade DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_causas, 1):
    causa_short = row['causa_acidente'][:50] + '...' if len(row['causa_acidente']) > 50 else row['causa_acidente']
    print(f"   {i}. {causa_short}")
    print(f"      {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes")

# Top 3 Condições Meteorológicas (ordenado por letalidade)
print(f"\n\n☁️ TOP 3 CONDIÇÕES METEOROLÓGICAS MAIS LETAIS:")
top_meteo = spark.sql("""
    SELECT 
        amb.condicao_metereologica,
        SUM(f.qtd_mortos) as total_mortos,
        COUNT(*) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_ambiente amb ON f.ambiente_sk = amb.ambiente_sk
    WHERE f.flag_fatal = 1
    GROUP BY amb.condicao_metereologica
    ORDER BY taxa_letalidade DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_meteo, 1):
    print(f"   {i}. {row['condicao_metereologica']}: {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes fatais")

# Top 3 Meses (ordenado por letalidade)
print(f"\n\n📆 TOP 3 MESES MAIS LETAIS:")
top_meses = spark.sql("""
    SELECT 
        CASE t.mes
            WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Março'
            WHEN 4 THEN 'Abril' WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho'
            WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto' WHEN 9 THEN 'Setembro'
            WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
        END as mes_nome,
        SUM(f.qtd_mortos) as total_mortos,
        COUNT(*) as acidentes_fatais,
        ROUND(100.0 * SUM(f.qtd_mortos) / NULLIF(SUM(f.qtd_mortos + f.qtd_feridos + f.qtd_ilesos), 0), 2) as taxa_letalidade
    FROM acidentes.gold.fato_acidentes f
    INNER JOIN acidentes.gold.dim_tempo t ON f.tempo_sk = t.tempo_sk
    WHERE f.flag_fatal = 1
    GROUP BY t.mes
    ORDER BY taxa_letalidade DESC
    LIMIT 3
""").collect()

for i, row in enumerate(top_meses, 1):
    print(f"   {i}. {row['mes_nome']}: {row['total_mortos']} mortos ({row['taxa_letalidade']}% letalidade) em {row['acidentes_fatais']} acidentes fatais")

print("\n" + "=" * 100)
print("✅ ANÁLISE CONCLUÍDA - Dados prontos para exportação!")
print("=" * 100)

---

# 🔒 GOVERNANÇA DE DADOS

## 📊 Pipeline de Metadados

O pipeline implementa **metadados de rastreabilidade** em todas as camadas:

### 🟊 **Bronze (Raw → Bronze)**
Adiciona metadados de ingestão:
* `ingestion_timestamp` - timestamp da ingestão
* `ingestion_date` - data da ingestão
* `ingestion_id` - ID único da ingestão
* `source_table` - tabela de origem

### 🥈 **Silver (Bronze → Silver)**
Remove metadados da Bronze (já cumpriram seu papel) e colunas não relevantes para análise.

### 🥇 **Gold (Silver → Gold)**
Adiciona novos metadados de exportação:
* `export_timestamp` - timestamp da exportação
* `export_date` - data da exportação
* `export_id` - ID único da exportação
* `source_table` - tabela Silver de origem
* `layer` - camada do pipeline (gold)

### 📝 **Tabela de Log de Ingestão**
Tabela centralizada: `acidentes.gold.ingestion_log`

Registra todas as execuções do pipeline:
* `log_id` - ID único da execução (UUID)
* `pipeline_name` - Nome do pipeline (ex: silver_to_gold)
* `layer` - Camada processada
* `execution_timestamp` - Quando foi executado
* `status` - SUCCESS / FAILED / RUNNING
* `records_processed` - Total de registros processados
* `tables_exported` - Número de tabelas exportadas
* `execution_time_seconds` - Tempo de execução
* `error_message` - Mensagem de erro (se houver)
* `user` - Usuário que executou

## 🎯 Benefícios

✅ **Rastreabilidade completa**: Saber quando e de onde vieram os dados  
✅ **Auditoria**: Compliance e histórico de execuções  
✅ **Monitoramento**: Acompanhar performance e taxa de sucesso  
✅ **Debug**: Facilita identificação de problemas  
✅ **Reprodução**: Recriar estado dos dados em qualquer momento  
✅ **SLA**: Métricas de tempo de execução e disponibilidade

---

In [0]:
# ========================================
# 📝 CRIAR TABELA DE LOG DE INGESTÃO
# Registra todas as execuções do pipeline para auditoria
# ========================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Criar schema da tabela de log
log_schema = StructType([
    StructField("log_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("execution_timestamp", TimestampType(), False),
    StructField("execution_date", StringType(), False),
    StructField("status", StringType(), False),
    StructField("records_processed", IntegerType(), True),
    StructField("tables_exported", IntegerType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_path", StringType(), True),
    StructField("execution_time_seconds", IntegerType(), True),
    StructField("error_message", StringType(), True),
    StructField("user", StringType(), True)
])

# Criar tabela se não existir
spark.sql("""
    CREATE TABLE IF NOT EXISTS acidentes.gold.ingestion_log (
        log_id STRING NOT NULL COMMENT 'ID único da execução (UUID)',
        pipeline_name STRING NOT NULL COMMENT 'Nome do pipeline executado',
        layer STRING NOT NULL COMMENT 'Camada do pipeline (bronze/silver/gold)',
        execution_timestamp TIMESTAMP NOT NULL COMMENT 'Timestamp da execução',
        execution_date STRING NOT NULL COMMENT 'Data da execução (YYYY-MM-DD)',
        status STRING NOT NULL COMMENT 'Status da execução (SUCCESS/FAILED/RUNNING)',
        records_processed INT COMMENT 'Total de registros processados',
        tables_exported INT COMMENT 'Número de tabelas exportadas',
        source_table STRING COMMENT 'Tabela de origem',
        target_path STRING COMMENT 'Caminho de destino',
        execution_time_seconds INT COMMENT 'Tempo de execução em segundos',
        error_message STRING COMMENT 'Mensagem de erro (se houver)',
        user STRING COMMENT 'Usuário que executou o pipeline'
    )
    USING DELTA
    COMMENT 'Log de execuções do pipeline de dados - Governança e Auditoria'
""")

print("✅ Tabela acidentes.gold.ingestion_log criada/verificada")
print("📝 Schema da tabela de log:")
spark.table("acidentes.gold.ingestion_log").printSchema()

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS acidentes.gold.exports")

In [0]:
# ========================================
# 💾 LOADING & DESTINATION
# Exportar tabelas Gold para Parquet e CSV (requisito do trabalho)
# Inclui metadados de governança para rastreabilidade
# ========================================

import os
from datetime import datetime
from pyspark.sql import functions as F
import uuid
import time

# Definir diretório de destino (usando Unity Catalog Volume)
# Nota: As tabelas Gold já estão persistidas no Unity Catalog
# Esta exportação é opcional para backup/compartilhamento externo
base_path = "/Volumes/acidentes/gold/exports"
gold_path = f"{base_path}/gold_data"
export_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_date = datetime.now().strftime("%Y-%m-%d")

# Metadados de governança (rastreabilidade)
ingestion_id = str(uuid.uuid4())
ingestion_ts = F.current_timestamp()
ingestion_date = F.current_date()

# Obter usuário atual
current_user = spark.sql("SELECT current_user() as user").collect()[0]['user']

# Iniciar cronômetro
start_time = time.time()

print("=" * 80)
print("💾 EXPORTANDO DADOS DA CAMADA GOLD COM METADADOS DE GOVERNANÇA")
print("=" * 80)
print(f"\n🔖 ID de Ingestão: {ingestion_id}")
print(f"🕒 Timestamp: {export_timestamp}\n")

# Lista de tabelas para exportar
tabelas_export = [
    ("acidentes.gold.dim_tempo", "dim_tempo", "acidentes.silver.acidentes_2025_clean"),
    ("acidentes.gold.dim_localizacao", "dim_localizacao", "acidentes.silver.acidentes_2025_clean"),
    ("acidentes.gold.dim_condicoes_via", "dim_condicoes_via", "acidentes.silver.acidentes_2025_clean"),
    ("acidentes.gold.dim_ambiente", "dim_ambiente", "acidentes.silver.acidentes_2025_clean"),
    ("acidentes.gold.dim_tipo_acidente", "dim_tipo_acidente", "acidentes.silver.acidentes_2025_clean"),
    ("acidentes.gold.fato_acidentes", "fato_acidentes", "acidentes.silver.acidentes_2025_clean")
]

# Registrar início da execução no log
try:
    spark.sql(f"""
        INSERT INTO acidentes.gold.ingestion_log VALUES (
            '{ingestion_id}',
            'silver_to_gold',
            'gold',
            current_timestamp(),
            '{export_date}',
            'RUNNING',
            NULL,
            {len(tabelas_export)},
            'acidentes.silver.acidentes_2025_clean',
            '{gold_path}',
            NULL,
            NULL,
            '{current_user}'
        )
    """)
    print("✅ Execução registrada no log de ingestão\n")
except Exception as e:
    print(f"⚠️ Aviso: Não foi possível registrar no log: {e}\n")

# Exportar cada tabela
total_records = 0
for tabela_full, nome_arquivo, source_table in tabelas_export:
    print(f"\n📦 Exportando {nome_arquivo}...")
    
    # Ler tabela
    df = spark.table(tabela_full)
    
    # Adicionar metadados de governança para rastreabilidade
    df_governanca = df \
        .withColumn("export_timestamp", ingestion_ts) \
        .withColumn("export_date", ingestion_date) \
        .withColumn("export_id", F.lit(ingestion_id)) \
        .withColumn("source_table", F.lit(source_table)) \
        .withColumn("layer", F.lit("gold"))
    
    # Caminho de destino
    output_path = f"{gold_path}/{nome_arquivo}"
    
    # Salvar como Parquet (formato otimizado) COM METADADOS DE GOVERNANÇA
    df_governanca.write.mode("overwrite").parquet(output_path)
    
    # Também salvar como CSV para facilitar visualização (SEM metadados para legibilidade)
    csv_path = f"{gold_path}/csv/{nome_arquivo}"
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(csv_path)
    
    count = df.count()
    total_records += count
    print(f"   ✅ Parquet (com governança): {output_path}")
    print(f"   ✅ CSV (dados puros): {csv_path}")
    print(f"   📊 Registros: {count:,}")

# Calcular tempo de execução
end_time = time.time()
execution_time = int(end_time - start_time)

# Atualizar log com sucesso
try:
    spark.sql(f"""
        UPDATE acidentes.gold.ingestion_log
        SET status = 'SUCCESS',
            records_processed = {total_records},
            execution_time_seconds = {execution_time}
        WHERE log_id = '{ingestion_id}'
    """)
except Exception as e:
    print(f"⚠️ Aviso: Não foi possível atualizar o log: {e}")

print("\n" + "=" * 80)
print("✅ EXPORTAÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 80)
print(f"⏱️ Tempo de execução: {execution_time} segundos")
print(f"📊 Total de registros processados: {total_records:,}")
print(f"\n📁 Localização dos arquivos:")
print(f"   • Parquet (com governança): {gold_path}/[nome_tabela]")
print(f"   • CSV (dados puros): {gold_path}/csv/[nome_tabela]")
print("\n🔖 Metadados de Governança incluídos no Parquet:")
print(f"   • export_timestamp: Timestamp da exportação")
print(f"   • export_date: Data da exportação")
print(f"   • export_id: {ingestion_id}")
print(f"   • source_table: Tabela de origem (Silver)")
print(f"   • layer: Camada do pipeline (gold)")
print("\n🎯 Dados prontos para consumo por:")
print("   • Ferramentas de visualização (Power BI, Tableau)")
print("   • Análises em Python/R")
print("   • Machine Learning")
print("   • Dashboards")
print("\n📊 DESTINO FINAL:")
print(f"   • Formato: Parquet (otimizado + governança) + CSV (legível)")
print(f"   • Localização: {gold_path}")
print(f"   • Status: Pronto para consumo e rastreável")

In [0]:
# ========================================
# 📝 CONSULTAR LOG DE INGESTÃO
# Verificar histórico de execuções
# ========================================

print("=" * 100)
print("📝 HISTÓRICO DE EXECUÇÕES DO PIPELINE")
print("=" * 100)

# Últimas 10 execuções
df_log = spark.sql("""
    SELECT 
        execution_timestamp,
        pipeline_name,
        layer,
        status,
        records_processed,
        tables_exported,
        execution_time_seconds,
        user,
        log_id
    FROM acidentes.gold.ingestion_log
    ORDER BY execution_timestamp DESC
    LIMIT 10
""")

print(f"\n📊 Total de execuções registradas: {spark.table('acidentes.gold.ingestion_log').count()}")
print("\n📅 Últimas 10 execuções:\n")
display(df_log)

# Estatísticas de execuções
print("\n\n📊 ESTATÍSTICAS:")
stats = spark.sql("""
    SELECT 
        COUNT(*) as total_execucoes,
        SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) as execucoes_sucesso,
        SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) as execucoes_falha,
        ROUND(100.0 * SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) / COUNT(*), 2) as taxa_sucesso,
        AVG(execution_time_seconds) as tempo_medio_segundos,
        SUM(records_processed) as total_registros_processados
    FROM acidentes.gold.ingestion_log
    WHERE status IN ('SUCCESS', 'FAILED')
""").collect()[0]

print(f"   • Total de execuções: {stats['total_execucoes']}")
print(f"   • Execuções com sucesso: {stats['execucoes_sucesso']}")
print(f"   • Execuções com falha: {stats['execucoes_falha']}")
print(f"   • Taxa de sucesso: {stats['taxa_sucesso']}%")
if stats['tempo_medio_segundos']:
    print(f"   • Tempo médio de execução: {int(stats['tempo_medio_segundos'])} segundos")
if stats['total_registros_processados']:
    print(f"   • Total de registros processados: {stats['total_registros_processados']:,}")

print("\n" + "=" * 100)